# Fairness Test

## Model A: sex and race included

In [1]:
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, recall_score, precision_score

In [2]:
data = pd.read_csv('data/adult.csv', header=None)
data.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
data = data[['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'race', 'sex', 'hours-per-week', 'capital-gain', 'capital-loss', 'income']]
data = data.replace('.*\\?.*', np.nan, regex=True).dropna()
data = pd.get_dummies(data, columns=['workclass', 'marital-status', 'occupation', 'race', 'sex'])

In [3]:
X = data.drop(['income'], axis=1)
y = data['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=21)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [4]:
model = MLPClassifier(random_state=21, max_iter=1000, hidden_layer_sizes=(50,), alpha=0.05).fit(X_train, y_train)

In [5]:
pred = model.predict(X_test) 

print(f'\n--- MODEL A: sex and race included   ({X_test.shape[1]} columns)')
print(f'{"group":8s} {"n":>5s} {"base":>6s} {"selected":>9s} {"precision":>10s} {"recall":>7s}')
rec = {}
for g in ['Male', 'Female']:
    m = (X_test[f"sex_ {g}"] > 0).values
    rec[g] = recall_score(y_test[m], pred[m], pos_label=" >50K")
    print(f'{g:8s} {m.sum():5d} {(y_test[m] == " >50K").mean():6.3f} {(pred[m] == " >50K").mean():9.3f} '
          f'{precision_score(y_test[m], pred[m], pos_label=" >50K"):10.3f} {rec[g]:7.3f}')
print(f'recall gap  {rec["Male"] - rec["Female"]:.3f}')


--- MODEL A: sex and race included   (40 columns)
group        n   base  selected  precision  recall
Male      9350  0.312     0.283      0.723   0.656
Female    4460  0.113     0.079      0.753   0.524
recall gap  0.133


## Model B: sex and race excluded

In [6]:
data = pd.read_csv('data/adult.csv', header=None)
data.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
data = data[['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'race', 'sex', 'hours-per-week', 'capital-gain', 'capital-loss', 'income']]
data = data.replace('.*\\?.*', np.nan, regex=True).dropna()
sex_data = data['sex'].str.strip()
data = data.drop(['race', 'sex'], axis=1)
data = pd.get_dummies(data, columns=['workclass', 'marital-status', 'occupation'])

In [7]:
X = data.drop(['income'], axis=1)
y = data['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=21)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [8]:
with open("models/mlp.pkl", "rb") as file:
    model = pickle.load(file)

In [9]:
y_pred = model.predict(X_test)
sex_test = sex_data.loc[X_test.index]

print(f'\n--- MODEL B: sex and race excluded   ({X_test.shape[1]} columns)')
print(f'{"group":8s} {"n":>5s} {"base":>6s} {"selected":>9s} {"precision":>10s} {"recall":>7s}')
rec = {}
for g in ['Male', 'Female']:
    m = (sex_test == g).values
    rec[g] = recall_score(y_test[m], y_pred[m], pos_label=" >50K")
    print(f'{g:8s} {m.sum():5d} {(y_test[m] == " >50K").mean():6.3f} {(y_pred[m] == " >50K").mean():9.3f} '
          f'{precision_score(y_test[m], y_pred[m], pos_label=" >50K"):10.3f} {rec[g]:7.3f}')
print(f'recall gap  {rec["Male"] - rec["Female"]:.3f}')


--- MODEL B: sex and race excluded   (33 columns)
group        n   base  selected  precision  recall
Male      9350  0.312     0.277      0.733   0.651
Female    4460  0.113     0.076      0.762   0.512
recall gap  0.139


The recall gap with sex and race excluded (0.139) is actually larger than the recall gap for the model that includes sex and race (0.133). This is likely because without the context of sex, the model needs to make assumptions based on the global majority instead of what it can learn to be true for each group.

# Leakage Test

## Model A: capital-gain and capital-loss included

In [10]:
data = pd.read_csv('data/adult.csv', header=None)
data.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
data = data[['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'hours-per-week', 'capital-gain', 'capital-loss', 'income']]
data = data.replace('.*\\?.*', np.nan, regex=True).dropna()
data = pd.get_dummies(data, columns=['workclass', 'marital-status', 'occupation'])

In [11]:
X = data.drop(['income'], axis=1)
y = data['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=21)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [12]:
with open("models/mlp.pkl", "rb") as file:
    model = pickle.load(file)

In [13]:
y_pred = model.predict(X_test) 
y_prob = model.predict_proba(X_test)[:, 1] 

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
print(classification_report(y_test, y_pred, digits=3))

auc = roc_auc_score(y_test, y_prob) 
print(f"AUC: {auc:.3f}")

Accuracy: 0.852
Confusion Matrix:
 [[9618  773]
 [1264 2155]]
              precision    recall  f1-score   support

       <=50K      0.884     0.926     0.904     10391
        >50K      0.736     0.630     0.679      3419

    accuracy                          0.852     13810
   macro avg      0.810     0.778     0.792     13810
weighted avg      0.847     0.852     0.848     13810

AUC: 0.907


## Model B: capital-gain and capital-loss excluded

In [14]:
data = pd.read_csv('data/adult.csv', header=None)
data.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
data = data[['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'hours-per-week', 'income']]
data = data.replace('.*\\?.*', np.nan, regex=True).dropna()
data = pd.get_dummies(data, columns=['workclass', 'marital-status', 'occupation'])

In [15]:
X = data.drop(['income'], axis=1)
y = data['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=21)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [16]:
model = MLPClassifier(random_state=21, max_iter=1000, hidden_layer_sizes=(50,), alpha=0.05).fit(X_train, y_train)

In [17]:
y_pred = model.predict(X_test) 
y_prob = model.predict_proba(X_test)[:, 1] 

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
print(classification_report(y_test, y_pred, digits=3))

auc = roc_auc_score(y_test, y_prob) 
print(f"AUC: {auc:.3f}")

Accuracy: 0.833
Confusion Matrix:
 [[9548  843]
 [1470 1949]]
              precision    recall  f1-score   support

       <=50K      0.867     0.919     0.892     10391
        >50K      0.698     0.570     0.628      3419

    accuracy                          0.833     13810
   macro avg      0.782     0.744     0.760     13810
weighted avg      0.825     0.833     0.827     13810

AUC: 0.886


When removing capital-gain and capital-loss, every single metric decreased significantly. You could argue that this is due to a leakage but a capital-gain of over 50K doesn't directly impact income. All it shows is that the person is in a comfortable enough position to be able to spend money on investing, which is a strong indicator that they earn greater than 50K.